# 03 · Detektor bola mati (`D`)

**Tujuan.** Untuk setiap frame yang dinilai, putuskan **bola hidup atau mati**. Tahap ini tidak peduli siapa yang
menguasai bola; itu urusan tahap H/A. Keluarannya juga menandai setiap **restart** (akhir segmen `D`) beserta
tim yang kemungkinan besar melakukannya.

| | |
|---|---|
| **Input** | `cache/prep.npz`, `cache/tags.csv` |
| **Output** | `cache/dead.npz`, `cache/dead_segments.csv`, `cache/restarts.csv`, `cache/dead_meta.json`, laporan `cache/reports/03_bola_mati.*` |
| **Waktu** | ± 1 menit |

### Riwayat versi

| Versi | Isi | GT publik (informasi) |
|---|---|---|
| v3 | fitur kolektif + Viterbi + bias dari prior | presisi ± 0,52, recall ± 0,91, 7 segmen vs 4 |
| v4 | v3 + fitur **aktivitas operan** + **jangkar** bukti hampir pasti (bagian 1b) | presisi ± 0,80, recall ± 0,81, 4 segmen (tepat) |
| v5 | v4 + **syarat isyarat henti**: segmen `D` tanpa satu pun isyarat henti dibuang; restart mencatat **tim pengambil** dari tag | presisi ± 0,85, recall ± 0,81, 3 segmen |
| **v6** | v5 dengan **broadcast**: bola broadcast mengisi celah (`01` §7b) + fitur & isyarat **replay/close-up** | lihat bagian 5 |

Mengapa v4: pada v3, ± 20% frame **perebutan bola** di seluruh match dicap `D` (di GT publik: 0%), ± 15% bola yang
jelas keluar lapangan tidak dicap `D`, dan sebagian besar `D` palsu menempel di tepi segmen — `D` terlalu panjang.

### Pendekatan

1. **Fitur kolektif yang halus dalam waktu** — semua dibuat sehingga "lebih besar = lebih mirip bola mati",
   lalu distandarkan (rata-rata 0, simpangan 1 di frame dinilai).
2. **Skor** = jumlah berbobot fitur + **bias**. Bobot ditetapkan dari prinsip (bukan dilatih di GT publik — 4 segmen
   terlalu sedikit; percobaan sebelumnya menunjukkan bobot terlatih kalah dari bobot manual pada validasi
   *leave-one-segment-out*).
3. **Viterbi 2 keadaan** (hidup/mati) dengan biaya setiap pergantian → segmen utuh, bukan keputusan per frame
   yang berkedip. Ingat prinsip spellF1: `D` palsu di tengah possession memecahnya jadi dua.
4. **Aturan pasti** yang memaksa `D`: babak 2 sebelum kick-off (FAQ.md), dan jeda sebelum kick-off setelah gol.
4b. **Jangkar** (v4): bukti yang hampir pasti diberi skor kuat per frame — perebutan bola → hidup, bola jelas di luar
   lapangan → mati, celah bola sangat panjang → mati. Jangkar tidak memaksa; ia hanya bukti yang sangat kuat,
   sehingga Viterbi tetap bisa menimbangnya terhadap bukti lain.
5. **Bias dipilih agar porsi `D` sesuai prior dari pengetahuan sepak bola** (waktu bermain efektif), bukan dari GT.

### Mengapa bias perlu prior?

Dengan fitur terstandar dan bias 0, Viterbi memotong kira-kira di "rata-rata frame" — porsi `D` yang dihasilkan
adalah kebetulan dari standardisasi, bukan bukti. Lebih jujur menyatakan asumsinya secara eksplisit:
waktu bermain efektif di LaLiga umumnya sekitar **50–56 menit** dari ± 95–100 menit pertandingan. Frame dinilai
± 105,9 menit (termasuk 2 menit sebelum kick-off babak 2), sehingga porsi `D` yang wajar ± **0,45–0,52**.
Prior ini *parameter yang dinyatakan*, dan bagian 6 menunjukkan seberapa sensitif hasil terhadapnya.

In [ ]:
%run ./common.ipynb
import warnings

## Parameter

| Parameter | Nilai | Alasan |
|---|---|---|
| `WEIGHTS` | lihat sel | urutan kekuatan bukti dari prinsip + temuan 02: pemain tenang paling kuat; celah bola panjang dan bola dekat garis sedang; bola pelan dan jendela set piece lemah |
| `SWITCH_COST` | 20 | biaya (dalam satuan skor) setiap pergantian hidup↔mati. Segmen harus "dibayar" oleh bukti ± 20 satuan |
| `MIN_SEG` | 50 frame (2 s) | segmen `D` lebih pendek dari ini dibuang (bola mati di sepak bola hampir selalu > 2 s) |
| `D_SHARE_PRIOR` | 0,48 | porsi `D` sasaran di frame dinilai (lihat atas) |
| `GAP_LONG_S` | 3 s | celah bola lebih pendek dari ini dianggap biasa (oklusi singkat), tidak menambah bukti |
| `PASS_V` | 8 m/s | bola lebih cepat dari ini di dalam lapangan = operan/tembakan → permainan hidup |
| `ANCHOR` | 3,0 | kekuatan jangkar per frame (dalam satuan simpangan baku fitur) — jauh di atas bukti biasa |
| `CONTEST_R`, `CONTEST_BV` | 2,0 m, 1 m/s | perebutan: pemain kedua tim ≤ 2 m dari bola yang bergerak (bola dipegang untuk lemparan ke dalam tidak dihitung) |
| `OOB_ANCHOR_M`, `OOB_ANCHOR_MIN` | 1,0 m, 25 frame | bola ≥ 1 m di luar garis selama ≥ 1 s = bola keluar |
| `GAP_ANCHOR_S` | 10 s | celah bola sepanjang ini = bola mati (dibuktikan tanpa GT di bagian 1b) |
| `REQUIRE_CUE` | ya | (v5) segmen `D` wajib memuat ≥ 1 isyarat henti (bagian 1c) |
| `CUE_STILL_SD`, `CUE_STILL_MIN` | 0,3 m, 25 frame | isyarat "bola diletakkan": simpangan posisi bola ± 0,5 s < 0,3 m selama ≥ 1 s |
| `CUE_OFFPITCH_S` | 3 s | (v6) isyarat "siaran menayangkan replay/close-up" selama ≥ 3 s |

In [ ]:
WEIGHTS = {"calm": 1.0, "no_pass": 1.0, "gap_long": 0.5, "near_line": 0.5, "bc_offpitch": 0.5, "ball_slow": 0.4,
           "in_sp": 0.3}
SWITCH_COST = 20.0
MIN_SEG = 50
D_SHARE_PRIOR = 0.48
GAP_LONG_S = 3.0
PASS_V = 8.0
ANCHOR = 3.0
CONTEST_R, CONTEST_BV = 2.0, 1.0
OOB_ANCHOR_M, OOB_ANCHOR_MIN = 1.0, 25
GAP_ANCHOR_S = 10.0
REQUIRE_CUE = True
CUE_STILL_SD, CUE_STILL_MIN = 0.3, 25
CUE_OFFPITCH_S = 3.0

In [ ]:
rep = Report("03_bola_mati")
CHECKS = []


def check(name, ok, detail="", critical=True):
    ok = bool(ok)
    status = "PASS" if ok else ("FAIL" if critical else "WARN")
    CHECKS.append({"cek": name, "status": status, "kritis": critical, "keterangan": detail})
    rep("[{}] {}{}".format(status, name, " — " + detail if detail else ""))
    return ok


z = load_prep()
tags = load_tags()
P, HOME = z["players"], z["slot_is_home"]
obs = z["ball_src"] > 0
bx, by = z["ball_filled"][:, 0], z["ball_filled"][:, 1]

## 1. Fitur

| Fitur | Definisi | Mengapa |
|---|---|---|
| `calm` | −(log kecepatan pemain rata-rata ± 3 s − log median lokal ± 2 menit) | pemain melambat saat bola mati; dibanding norma lokal agar kelelahan/tempo tidak menipu (AUC 02: ± 0,84) |
| `gap_long` | log(1 + panjang celah dalam detik) untuk frame di dalam celah ≥ `GAP_LONG_S`, selain itu 0 | bola hidup hampir selalu terlihat; celah panjang mengarah ke bola mati (temuan 02 #2) |
| `near_line` | −min jarak bola ke garis lapangan dalam ± 2 s (dibatasi −5…20 m; bola hilang → 20 m) | restart terjadi di garis: lemparan, sepak pojok, tendangan gawang (± 0,77) |
| `ball_slow` | −log(1 + median kecepatan bola ± 2 s) | bola diletakkan/diam sebelum restart (± 0,71); netral bila bola hilang |
| `in_sp` | frame di dalam jendela tag `SET PIECES` | petunjuk lemah: jendela tag berdurasi tetap, bukan durasi bola mati |
| `no_pass` | −(porsi frame dengan bola > `PASS_V` di dalam lapangan, ± 2 s) | operan dan tembakan hanya terjadi saat bola hidup (AUC publik ± 0,84, setara `calm`) |
| `bc_offpitch` | (v6) siaran broadcast pada frame itu tidak menampilkan lapangan (homografi tidak valid: replay, close-up, penonton) | sutradara siaran memutar replay/close-up saat permainan berhenti. **Tanpa GT:** di frame ini pemain bergerak ± 1,0 m/s (seperti bola mati pasti), di siaran lapangan ± 2,1 m/s. Tidak hampir pasti (close-up juga terjadi saat bola hidup) → fitur, bukan jangkar |

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    ps = np.nanmean(P[..., 2], axis=1)
sm = pd.Series(ps).rolling(151, center=True, min_periods=50).mean()
loc = sm.rolling(6001, center=True, min_periods=500).median()
calm = -(np.log(sm + 0.1) - np.log(loc + 0.1)).to_numpy()

gap_len = np.zeros(N_FRAMES)
for h, (a, b) in HALVES.items():
    s, e = runs(~obs[a:b + 1])
    for si, ei in zip(s, e):
        gap_len[a + si:a + ei] = (ei - si) / FPS
gap_long = np.where(gap_len >= GAP_LONG_S, np.log1p(gap_len), 0.0)

line = np.where(obs, np.minimum.reduce([bx, PITCH_L - bx, by, PITCH_W - by]), np.nan)
mn = pd.Series(line).rolling(101, center=True, min_periods=10).min().to_numpy()
near_line = -np.clip(np.nan_to_num(mn, nan=20.0), -5.0, 20.0)

h = 12
ball_v = np.full(N_FRAMES, np.nan)
ball_v[h:-h] = np.hypot(bx[2 * h:] - bx[:-2 * h], by[2 * h:] - by[:-2 * h]) / (2 * h / FPS)
ball_slow = -np.log1p(pd.Series(ball_v).rolling(101, center=True, min_periods=20).median().to_numpy())

inside = np.nan_to_num(z["ball_oob_m"], nan=1.0) == 0
fast = obs & inside & (np.nan_to_num(ball_v) > PASS_V)
no_pass = -pd.Series(fast.astype(float)).rolling(101, center=True, min_periods=1).mean().to_numpy()
bc_offpitch = (z["bc_view"] == -1).astype(float)
rep("siaran broadcast replay/close-up: {:.1f}% frame dinilai".format(100 * bc_offpitch[GRADED].mean()))

sp = tags[tags.code == "SET PIECES"]
in_sp = np.zeros(N_FRAMES)
for r in sp.itertuples():
    in_sp[max(r.start_f, 0):min(r.end_f, N_FRAMES)] = 1.0

FEATS = list(WEIGHTS)
F = np.stack([dict(calm=calm, no_pass=no_pass, gap_long=gap_long, near_line=near_line, bc_offpitch=bc_offpitch,
                   ball_slow=ball_slow, in_sp=in_sp)[k]
              for k in FEATS], axis=1)
mu = np.nanmean(F[GRADED], axis=0)
sd = np.nanstd(F[GRADED], axis=0) + 1e-9
Zf = (F - mu) / sd
Zf[~np.isfinite(Zf)] = 0.0          # fitur tak terukur = netral
W = np.array([WEIGHTS[k] for k in FEATS])
feat_score = Zf @ W
rep("fitur: {} | bobot: {}".format(FEATS, W.tolist()))
rep("frame dalam celah bola ≥ {:.0f} s: {:.1f}% dari frame dinilai".format(
    GAP_LONG_S, 100 * (gap_len[GRADED] >= GAP_LONG_S).mean()))

## 1b. Jangkar: bukti yang hampir pasti

| Jangkar | Definisi | Arah | Bukti bahwa ia hampir pasti |
|---|---|---|---|
| `contest` | bola terlihat & bergerak > `CONTEST_BV`, pemain **kedua** tim ≤ `CONTEST_R` darinya | hidup | GT publik: 0% frame perebutan adalah `D` (553 frame) |
| `oob1` | bola ≥ `OOB_ANCHOR_M` di luar garis selama ≥ 1 s | mati | GT publik: 100% `D`; secara definisi bola keluar = mati |
| `gap_anchor` | frame di dalam celah bola ≥ `GAP_ANCHOR_S` | mati | **tanpa GT**, tabel di bawah: di celah panjang pemain bergerak selambat (bahkan lebih lambat dari) bola mati yang pasti |

Skor akhir per frame = skor fitur + `ANCHOR` × (`oob1` + `gap_anchor` − `contest`).

Tabel di bawah membandingkan perilaku pemain di dalam celah bola menurut panjangnya, dengan dua kelompok
yang pasti mati (sebelum kick-off babak 2; bola jelas keluar) dan satu yang pasti hidup (perebutan). Bila celah
panjang menyerupai kelompok mati, jangkar celah sah — tanpa melihat GT.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    dH = np.where(obs, np.nanmin(np.hypot(P[:, HOME, 0] - bx[:, None], P[:, HOME, 1] - by[:, None]), axis=1), np.nan)
    dA = np.where(obs, np.nanmin(np.hypot(P[:, ~HOME, 0] - bx[:, None], P[:, ~HOME, 1] - by[:, None]), axis=1), np.nan)
contest = obs & (np.nan_to_num(dH, nan=99) <= CONTEST_R) & (np.nan_to_num(dA, nan=99) <= CONTEST_R) \
    & (np.nan_to_num(ball_v) > CONTEST_BV)
oob1 = np.zeros(N_FRAMES, dtype=bool)
for a, b in zip(*runs(np.nan_to_num(z["ball_oob_m"]) > OOB_ANCHOR_M)):
    if b - a >= OOB_ANCHOR_MIN:
        oob1[a:b] = True
gap_anchor = GRADED & (gap_len >= GAP_ANCHOR_S)

p90 = pd.Series(np.nanpercentile(P[..., 2], 90, axis=1)).rolling(75, center=True, min_periods=10).mean().to_numpy()
groups = {"pasti mati: sebelum kick-off babak 2": z["dead_prior"], "pasti mati: bola keluar ≥ 1 s": oob1 & GRADED,
          "pasti hidup: perebutan bola": contest & GRADED,
          "celah 3–10 s": GRADED & (gap_len >= 3) & (gap_len < 10), "celah 10–30 s": GRADED & (gap_len >= 10) & (gap_len < 30),
          "celah 30–60 s": GRADED & (gap_len >= 30) & (gap_len < 60), "celah ≥ 60 s": GRADED & (gap_len >= 60)}
beh = pd.DataFrame([{"kelompok": k, "frame": int(m.sum()),
                     "kecepatan pemain ± 3 s (median)": round(float(np.nanmedian(sm.to_numpy()[m])), 2),
                     "p90 kecepatan pemain (median)": round(float(np.nanmedian(p90[m])), 2)}
                    for k, m in groups.items()]).set_index("kelompok")
rep.table(beh, "perilaku pemain per kelompok (tanpa GT):")
n_gap_anchor = sum(1 for a, b in zip(*runs(gap_anchor)))
rep("jangkar: perebutan {} frame | bola keluar ≥1 s {} frame | celah ≥ {:.0f} s {} frame ({} celah, {:.1f}% frame dinilai)".format(
    int((contest & GRADED).sum()), int((oob1 & GRADED).sum()), GAP_ANCHOR_S, int(gap_anchor.sum()), n_gap_anchor,
    100 * gap_anchor[GRADED].mean()))

anchor_score = ANCHOR * (oob1.astype(float) + gap_anchor.astype(float) - contest.astype(float))
base_score = feat_score + anchor_score


xo_ = pd.Series(np.where(obs, bx, np.nan))
yo_ = pd.Series(np.where(obs, by, np.nan))
ball_sd = np.hypot(xo_.rolling(25, center=True, min_periods=15).std(),
                   yo_.rolling(25, center=True, min_periods=15).std()).to_numpy()
still_cs = np.concatenate(([0], np.cumsum(np.nan_to_num(ball_sd, nan=99.0) < CUE_STILL_SD)))
sp_start = np.zeros(N_FRAMES, dtype=bool)
for r in sp.itertuples():
    sp_start[min(max(int(r.start_f), 0), N_FRAMES - 1)] = True
offpitch_long = np.zeros(N_FRAMES, dtype=bool)
for a, b in zip(*runs(z["bc_view"] == -1)):
    if b - a >= CUE_OFFPITCH_S * FPS:
        offpitch_long[a:b] = True
cue_cs = np.concatenate(([0], np.cumsum((gap_len >= GAP_LONG_S) | oob1 | sp_start | offpitch_long)))


def has_cue(a, b):
    """Isyarat henti di [a, b): celah bola >= GAP_LONG_S, bola keluar >= 1 s, awal tag set piece, siaran replay/close-up
    >= CUE_OFFPITCH_S, bola diam >= 1 s, atau frame dipaksa."""
    return bool(cue_cs[b] - cue_cs[a] > 0 or still_cs[b] - still_cs[a] >= CUE_STILL_MIN or forced[a:b].any())


def anchor_proxies(mask):
    return {"perebutan dicap D %": round(100 * float(mask[contest & GRADED].mean()), 1),
            "bola keluar dicap D %": round(100 * float(mask[oob1 & GRADED].mean()), 1),
            "celah ≥ 10 s dicap D %": round(100 * float(mask[gap_anchor].mean()), 1)}

## 1c. Syarat isyarat henti (v5)

Bola mati selalu didahului **sesuatu yang menghentikan permainan**. Sebuah segmen `D` dipertahankan hanya bila
memuat minimal satu isyarat:

| Isyarat | Definisi |
|---|---|
| celah bola | bola tak terlihat ≥ `GAP_LONG_S` |
| bola keluar | jangkar `oob1` |
| bola diletakkan | bola nyaris diam (simpangan < `CUE_STILL_SD`) selama ≥ `CUE_STILL_MIN` frame |
| tag set piece | awal jendela tag `SET PIECES` di dalam segmen |
| siaran replay/close-up (v6) | broadcast tidak menampilkan lapangan ≥ `CUE_OFFPITCH_S` |
| aturan pasti | frame dipaksa (bagian 2) |

**Bukti tanpa GT:** pada v4 hanya ± 5 dari ± 114 segmen yang tak punya isyarat apa pun; di dalamnya pemain bergerak
± 1,5 m/s — jauh lebih cepat dari segmen beristirahat (± 1,1 m/s) dan mendekati permainan hidup. Syarat ini
dijalankan di dalam decoder (bagian 3), sehingga bisection porsi `D` sudah memperhitungkannya.

## 2. Aturan pasti

1. **Babak 2 sebelum kick-off** (`dead_prior`, 2.934 frame) — FAQ.md.
2. **Sebelum kick-off setelah gol.** Tag `Kick Off (After Goal)` berupa jendela 5 s. Untuk dua kick-off awal
   babak, waktu kick-off sebenarnya **dipublikasikan** (11.185 dan 113.742), sehingga jarak "awal tag → kick-off"
   bisa diukur tanpa GT. Jarak rata-rata itu dipakai untuk memperkirakan kick-off setelah gol, dan frame dari awal
   tag sampai perkiraan kick-off dipaksa `D`. Aturannya sengaja konservatif (hanya beberapa detik); selebrasi
   panjang sebelumnya diserahkan pada fitur.

In [ ]:
ko_tags = sp[sp.Type.str.lower().str.startswith("kick off")]
start_tags = ko_tags[ko_tags.Type.str.lower() == "kick off (start)"].sort_values("start_f")
offsets = [KICKOFF[h] - int(r.start_f) for h, r in zip((1, 2), start_tags.itertuples())]
KO_OFFSET = int(round(np.mean(offsets)))
rep("kick-off awal babak: awal tag → kick-off = {} frame → dipakai {} frame ({:.1f} s)".format(
    offsets, KO_OFFSET, KO_OFFSET / FPS))

forced = z["dead_prior"].copy()
goal_rows = []
for r in ko_tags[ko_tags.Type.str.lower() == "kick off (after goal)"].itertuples():
    ko_est = int(r.start_f) + KO_OFFSET
    forced[int(r.start_f):ko_est] = True
    goals_before = tags[(tags.code == "GOALS") & (tags.start_f < r.start_f)]
    last_goal = goals_before.start_f.max() if len(goals_before) else np.nan
    goal_rows.append({"kick-off (perkiraan)": ko_est, "babak": int(HALF[ko_est]),
                      "jarak dari tag GOALS terakhir (s)": round((ko_est - last_goal) / FPS, 1)})
goal_df = pd.DataFrame(goal_rows)
rep.table(goal_df, "kick-off setelah gol:")
forced &= GRADED
rep("frame dipaksa D: {} (prior babak 2 {} + setelah gol {})".format(
    int(forced.sum()), int(z["dead_prior"].sum()), int(forced.sum() - z["dead_prior"].sum())))

## 3. Decoder: Viterbi 2 keadaan + bias dari prior

Untuk satu nilai bias *b*: skor = `base_score + b`; emisi hidup = −skor/2, mati = +skor/2; frame dipaksa diberi
emisi ±10⁴. Viterbi dijalankan per babak (tidak melintasi jeda turun minum), lalu segmen `D` < `MIN_SEG` yang
tidak dipaksa dibuang.

Bias dicari dengan **bisection** sehingga porsi `D` di frame dinilai = `D_SHARE_PRIOR`. Porsi `D` naik monoton
terhadap *b*, jadi bisection selalu konvergen.

In [ ]:
TRANS = np.array([[0.0, SWITCH_COST], [SWITCH_COST, 0.0]])


def decode(bias):
    s = base_score + bias
    emis = np.stack([-s / 2, s / 2], axis=1)
    emis[forced, 0], emis[forced, 1] = -1e4, 1e4
    mask = np.zeros(N_FRAMES, dtype=bool)
    for h, (a, b) in HALVES.items():
        mask[a:b + 1] = viterbi(emis[a:b + 1], TRANS) == 1
    starts, ends = runs(mask)
    for a, b in zip(starts, ends):
        if b - a < MIN_SEG and not forced[a:b].any():
            mask[a:b] = False
    if REQUIRE_CUE:
        for a, b in zip(*runs(mask)):
            if not has_cue(a, b):
                mask[a:b] = False
    return mask


def d_share(mask):
    return float(mask[GRADED].mean())


with timer("bias 0 (referensi)"):
    share_b0 = d_share(decode(0.0))
rep("porsi D dengan bias 0: {:.3f}".format(share_b0))

lo, hi = -6.0, 6.0
with timer("bisection bias"):
    for _ in range(18):
        mid = (lo + hi) / 2
        if d_share(decode(mid)) < D_SHARE_PRIOR:
            lo = mid
        else:
            hi = mid
BIAS = round((lo + hi) / 2, 4)
d_mask = decode(BIAS)
rep("bias terpilih {:+.4f} → porsi D {:.3f} (sasaran {:.2f})".format(BIAS, d_share(d_mask), D_SHARE_PRIOR))
check("porsi D mengenai prior (±0,01)", abs(d_share(d_mask) - D_SHARE_PRIOR) <= 0.01, "{:.3f}".format(d_share(d_mask)))
check("semua frame dipaksa = D", d_mask[forced].all())
check("D hanya di frame dinilai", not d_mask[~GRADED].any())
prox = anchor_proxies(d_mask)
rep("proxy jangkar (seluruh match): {}".format(prox))

## 4. Segmen `D` dan restart

Setiap segmen `D` berakhir di sebuah **restart**. Untuk setiap restart dicatat:

- jenis & tim tag `SET PIECES` terdekat (bila ada),
- **tim terdekat ke bola di 2 s terakhir segmen `D`** — temuan 02 #9: tim yang melakukan restart adalah tim yang
  berdiri di dekat bola. Bila bola tak terlihat di sana, dipakai 2 s pertama setelah restart; bila tetap tak
  terlihat → `NA`.
- **(v5) tim pengambil `taker`** = `Team` tag `Corner Kick` / `Goal Kick` / `Free Kick` terdekat. Arti `Team` = tim
  pengambil diverifikasi **tanpa GT** dengan geometri: pada setiap sepak pojok bola berada di separuh lawan tim di
  tag (12/12), pada tendangan gawang di separuh sendiri (± 80%). Sebaliknya "tim terdekat di akhir `D`" ternyata
  tidak andal untuk sepak pojok (pemain bertahan berkerumun di dekat bola) — karena itu decoder (`06`) hanya
  memakai `taker`.

In [ ]:
near = np.where(np.isfinite(dH) & np.isfinite(dA), np.where(dH < dA, 1, -1), 0)


def nearest_team(a, b, min_visible=10):
    nn = near[max(a, 0):b]
    vis = nn != 0
    if vis.sum() < min_visible:
        return "NA", int(vis.sum())
    return ("H" if (nn[vis] == 1).mean() >= 0.5 else "A"), int(vis.sum())


seg_rows = []
for i, (a, b) in enumerate(zip(*runs(d_mask))):
    near_sp = sp[(sp.start_f < b + 300) & (sp.end_f > a)]
    tag_type, tag_side = "NA", "NA"
    if len(near_sp):
        row = near_sp.iloc[(near_sp.start_f - a).abs().argmin()]
        tag_type, tag_side = str(row.Type), str(row.side)
    team, n_vis = nearest_team(b - 50, b)
    source = "akhir D"
    if team == "NA":
        team, n_vis = nearest_team(b, b + 50)
        source = "awal hidup" if team != "NA" else "-"
    seg_rows.append({"seg": i, "half": int(HALF[a]), "start": int(a), "end_excl": int(b), "length": int(b - a),
                     "forced": bool(forced[a:b].any()), "tag_type": tag_type, "tag_side": tag_side,
                     "restart_team": team, "restart_source": source,
                     "taker": tag_side if tag_type in ("Corner Kick", "Goal Kick", "Free Kick") and tag_side in ("H", "A")
                     else "NA"})
segs = pd.DataFrame(seg_rows)
L = segs.length[~segs.forced]
rep("segmen D: {} | median {:.1f} s | p10 {:.1f} s | p90 {:.1f} s | maks {:.1f} s".format(
    len(segs), L.median() / FPS, L.quantile(0.1) / FPS, L.quantile(0.9) / FPS, L.max() / FPS))
rep("per babak: {}".format(segs.half.value_counts().sort_index().to_dict()))
rep("tim restart: {} | sumber: {}".format(segs.restart_team.value_counts().to_dict(),
                                          segs.restart_source.value_counts().to_dict()))
both = segs[(segs.tag_side.isin(["H", "A"])) & (segs.restart_team.isin(["H", "A"]))]
rep("tim pengambil dari tag (corner/goal kick/free kick): {}".format(segs.taker.value_counts().to_dict()))
rep("restart dengan tag bertim & tim terdekat terukur: {} | sepakat {}".format(
    len(both), int((both.tag_side == both.restart_team).sum())))
sp_nk = sp[~sp.Type.str.lower().str.startswith("kick off")]
hit = sum(bool(d_mask[max(r.start_f, 0):min(r.end_f, N_FRAMES)].any()) for r in sp_nk.itertuples())
rep("jendela tag set piece (bukan kick-off) yang memuat D: {} / {}".format(hit, len(sp_nk)))

## 5. Cek kebenaran pada GT publik

**Bukan** penilaian kualitas untuk memilih parameter (4 segmen). Bobot tidak dilatih di sini, jadi angka ini
adalah uji di luar sampel — tetapi tetap sangat kecil.

- presisi / recall per frame untuk `D` di jendela publik,
- jumlah segmen `D` prediksi vs GT,
- IoU terbaik setiap segmen `D` GT.

In [ ]:
gt = pm.read_label_csv(str(GT_PUBLIC))["label"].astype(str).str.strip().str.upper().to_numpy()
known = np.flatnonzero(gt != "X")
W0, W1 = int(known[0]), int(known[-1]) + 1


def window_metrics(mask):
    p, g = mask[W0:W1], gt[W0:W1] == "D"
    tp = int((p & g).sum())
    ps, pe = runs(p)
    ious = []
    for a, b in zip(*runs(g)):
        best = 0.0
        for c, d in zip(ps, pe):
            inter = max(0, min(b, d) - max(a, c))
            best = max(best, inter / float((b - a) + (d - c) - inter))
        ious.append(round(best, 2))
    return {"presisi": round(tp / max(int(p.sum()), 1), 2), "recall": round(tp / max(int(g.sum()), 1), 2),
            "segmen prediksi": len(ps), "segmen GT": len(ious), "IoU per segmen GT": ious}


pub = window_metrics(d_mask)
rep("GT publik: {}".format(pub))

## 6a. Ablation: sumbangan tiap perubahan v4

Setiap varian memakai prior porsi `D` yang sama (bias dicari ulang), sehingga yang berbeda hanya **di mana** `D`
diletakkan. Proxy jangkar dihitung di seluruh match tanpa GT; kolom publik hanya informasi.

In [ ]:
def fit_bias(score, target=D_SHARE_PRIOR, iters=16):
    global base_score
    saved, base_score = base_score, score
    lo_, hi_ = -8.0, 8.0
    for _ in range(iters):
        mid = (lo_ + hi_) / 2
        lo_, hi_ = (mid, hi_) if d_share(decode(mid)) < target else (lo_, mid)
    m = decode((lo_ + hi_) / 2)
    base_score = saved
    return m


W_v3 = np.array([0.0 if k in ("no_pass", "bc_offpitch") else WEIGHTS[k] for k in FEATS])
W_v5 = np.array([0.0 if k == "bc_offpitch" else WEIGHTS[k] for k in FEATS])
variants = {"v3 (tanpa operan, tanpa jangkar)": (Zf @ W_v3, False),
            "v3 + operan": (Zf @ W_v5, False),
            "v3 + jangkar": (Zf @ W_v3 + anchor_score, False),
            "v4 = v3 + operan + jangkar": (Zf @ W_v5 + anchor_score, False),
            "v5 = v4 + syarat isyarat henti": (Zf @ W_v5 + anchor_score, True),
            "v6 = v5 + fitur replay/close-up broadcast": (base_score, True)}
abl = []
with timer("ablation"):
    for name, (sc, cue) in variants.items():
        REQUIRE_CUE = cue
        m = fit_bias(sc)
        w = window_metrics(m)
        abl.append({"varian": name, "segmen D": len(runs(m)[0]), **anchor_proxies(m),
                    "publik presisi": w["presisi"], "publik recall": w["recall"], "publik segmen": w["segmen prediksi"]})
REQUIRE_CUE = True
abl = pd.DataFrame(abl).set_index("varian")
rep.table(abl, "ablation (prior porsi D sama):")

## 6b. Sensitivitas terhadap prior dan biaya pergantian

Bila hasil berubah drastis saat parameter digeser sedikit, parameter itu rapuh. Tabel ini **tidak** dipakai
untuk memilih nilai; ia menunjukkan seberapa besar keputusan bergantung padanya. Kolom publik hanya informasi.

In [ ]:
sens = []
for prior in (0.42, 0.45, 0.48, 0.51, 0.54):
    lo_, hi_ = -6.0, 6.0
    for _ in range(14):
        mid = (lo_ + hi_) / 2
        lo_, hi_ = (mid, hi_) if d_share(decode(mid)) < prior else (lo_, mid)
    m = decode((lo_ + hi_) / 2)
    w = window_metrics(m)
    sens.append({"prior D": prior, "bias": round((lo_ + hi_) / 2, 2), "segmen D": len(runs(m)[0]), **anchor_proxies(m),
                 "median D (s)": round(float(np.median(np.diff(np.stack(runs(m)), axis=0))) / FPS, 1),
                 "publik presisi": w["presisi"], "publik recall": w["recall"], "publik segmen": w["segmen prediksi"]})
sens = pd.DataFrame(sens).set_index("prior D")
rep.table(sens, "sensitivitas terhadap D_SHARE_PRIOR (SWITCH_COST tetap):")

sens_c = []
for cost in (10.0, 20.0, 40.0):
    TRANS[:] = [[0.0, cost], [cost, 0.0]]
    m = decode(BIAS)
    sens_c.append({"switch cost": cost, "porsi D": round(d_share(m), 3), "segmen D": len(runs(m)[0]), **anchor_proxies(m)})
anchor_bak = ANCHOR
for a_ in (1.0, 6.0):
    base_score = feat_score + anchor_score * (a_ / anchor_bak)
    m = fit_bias(base_score)
    w = window_metrics(m)
    sens_c.append({"switch cost": "ANCHOR={}".format(a_), "porsi D": round(d_share(m), 3), "segmen D": len(runs(m)[0]),
                   **anchor_proxies(m), "publik presisi": w["presisi"], "publik recall": w["recall"]})
base_score = feat_score + anchor_score
TRANS[:] = [[0.0, SWITCH_COST], [SWITCH_COST, 0.0]]
rep.table(pd.DataFrame(sens_c).set_index("switch cost"), "sensitivitas SWITCH_COST (bias tetap) dan ANCHOR (bias dicari ulang):")

## 7. Simpan

In [ ]:
np.savez(CACHE / "dead.npz", d_score=(base_score + BIAS).astype(np.float32), d_mask=d_mask, forced=forced,
         near_team=near.astype(np.int8))
segs.to_csv(CACHE / "dead_segments.csv", index=False)
segs[["seg", "end_excl", "tag_type", "tag_side", "restart_team", "restart_source", "taker"]].rename(
    columns={"end_excl": "restart_frame"}).to_csv(CACHE / "restarts.csv", index=False)
(CACHE / "dead_meta.json").write_text(json.dumps({
    "features": FEATS, "weights": WEIGHTS, "mu": mu.tolist(), "sd": sd.tolist(), "bias": BIAS,
    "switch_cost": SWITCH_COST, "min_seg": MIN_SEG, "d_share_prior": D_SHARE_PRIOR, "gap_long_s": GAP_LONG_S,
    "ko_offset": KO_OFFSET, "d_share": d_share(d_mask), "n_segments": len(segs), "version": "v6",
    "anchor": ANCHOR, "pass_v": PASS_V, "gap_anchor_s": GAP_ANCHOR_S, "contest_r": CONTEST_R}, indent=2), encoding="utf-8")

summary_df = pd.DataFrame(CHECKS)
n_fail = int((summary_df.status == "FAIL").sum())
rep("\nPASS {} | WARN {} | FAIL {}".format(int((summary_df.status == "PASS").sum()),
                                         int((summary_df.status == "WARN").sum()), n_fail))
rep.save({"checks": CHECKS, "bias": BIAS, "d_share": d_share(d_mask), "share_bias0": share_b0,
          "n_segments": len(segs), "public": pub, "anchor_proxies": prox, "ablation": abl.reset_index().to_dict("records"),
          "gap_behaviour": beh.reset_index().to_dict("records"), "sensitivity_prior": sens.reset_index().to_dict("records"),
          "sensitivity_cost": sens_c})
assert n_fail == 0

## Ringkasan & keputusan (v6)

**Ablation (prior porsi `D` sama, ± 0,48; semua varian memakai bola gabungan tactical + broadcast)**

| Varian | perebutan dicap `D` | bola keluar dicap `D` | celah ≥ 10 s dicap `D` | publik presisi / recall / segmen (GT 4) |
|---|---|---|---|---|
| v3 | ± 19% | ± 85% | ± 80% | 0,52 / 0,91 / 7 |
| v3 + operan | ± 16% | ± 86% | ± 89% | 0,64 / 0,84 / 8 |
| v3 + jangkar | ± 1% | ± 96% | ± 100% | 0,74 / 0,84 / 4 |
| v4 = + operan + jangkar | ± 1,4% | ± 98% | ± 100% | 0,80 / 0,81 / 4 |
| v5 = v4 + syarat isyarat henti | ± 1,5% | ± 98% | ± 100% | 0,82 / 0,86 / 5 |
| **v6** = v5 + fitur replay/close-up broadcast | **± 1,0%** | ± 97% | ± 100% | **0,86 / 0,86 / 4** |

**Bacaan**

1. **Jangkar adalah perubahan utama** (v3 → v4): tiga proxy tanpa GT membaik serentak, dan GT publik setuju.
2. **Jangkar celah panjang sah tanpa GT:** di celah bola ≥ 10 s pemain bergerak selambat — bahkan lebih lambat dari —
   kelompok yang pasti mati, jauh dari kelompok pasti hidup.
3. **Syarat isyarat henti (v5)** membuang segmen `D` tanpa bukti henti apa pun.
4. **Siaran broadcast adalah saksi kedua (v6).** Saat siaran menayangkan replay/close-up, pemain bergerak seperti saat
   bola mati — sutradara siaran mengisi jeda permainan. Fitur ini menurunkan perebutan yang dicap `D`, dan di jendela
   publik `D` pendek (5,5 s) yang sebelumnya terlewat kini tertangkap: 4 segmen, presisi = recall ± 0,86.
5. **Arti `Team` pada tag set piece = tim pengambil** (geometri) → kolom `taker` untuk prior restart di `06`.
6. Prior porsi `D` 0,48 tetap; kekuatan jangkar dan biaya pergantian tidak mengubah gambaran (bagian 6b).

**Masalah yang tersisa**

- Bobot fitur masih manual (dari prinsip); bobot `bc_offpitch` = 0,5 setara bukti sedang lainnya.
- Lemparan ke dalam (± 26) tidak punya tim di tag; tim pengambilnya belum diketahui dengan andal.

**Dipakai oleh tahap berikutnya:** `cache/dead.npz` (`d_mask`, `d_score`, `near_team`) dan `cache/restarts.csv`
(`restart_frame`, `taker`, `restart_team`).